# 01 Data Ingestion

Day 1 notebook for sourcing Bluestock mutual fund datasets from AMFI public sources and validating the raw ingestion layer.

## Ingestion Methodology

1. Identify the canonical AMFI data sources and raw project files.
2. Read all ten raw CSV datasets from `../data/raw/`.
3. Run structural diagnostics on each dataset with shape, dtypes, and the first five rows.
4. Verify AMFI scheme-code coverage between the fund master and historical NAV files.
5. Poll live NAV JSON from the public AMFI API and archive immutable backups for recovery.

## Source Inventory

The ingestion layer uses the following raw datasets:

- `01_fund_master.csv`
- `02_nav_history.csv`
- `03_aum_by_fund_house.csv`
- `04_monthly_sip_inflows.csv`
- `05_category_inflows.csv`
- `06_industry_folio_count.csv`
- `07_scheme_performance.csv`
- `08_investor_transactions.csv`
- `09_portfolio_holdings.csv`
- `10_benchmark_indices.csv`

In [1]:
from pathlib import Path
from datetime import datetime
import json

import pandas as pd
import requests

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 120)

RAW_DIR = Path('../data/raw')
if not RAW_DIR.exists():
    RAW_DIR = Path('data/raw')

API_BACKUP_DIR = RAW_DIR / 'api_backups'
API_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FILES = {
    'fund_master': '01_fund_master.csv',
    'nav_history': '02_nav_history.csv',
    'aum_by_fund_house': '03_aum_by_fund_house.csv',
    'monthly_sip_inflows': '04_monthly_sip_inflows.csv',
    'category_inflows': '05_category_inflows.csv',
    'industry_folio_count': '06_industry_folio_count.csv',
    'scheme_performance': '07_scheme_performance.csv',
    'investor_transactions': '08_investor_transactions.csv',
    'portfolio_holdings': '09_portfolio_holdings.csv',
    'benchmark_indices': '10_benchmark_indices.csv',
}

In [2]:
def load_raw_dataset(label: str, filename: str) -> pd.DataFrame:
    file_path = RAW_DIR / filename
    print(f'Loading [{label}] from {file_path}')
    return pd.read_csv(file_path)


def structural_check(label: str, df: pd.DataFrame) -> None:
    print()
    print('=' * 90)
    print(f'DATASET: {label}')
    print(f'Shape: {df.shape}')
    print('Dtypes:')
    print(df.dtypes.to_string())
    print('Head(5):')
    print(df.head(5).to_string(index=False))


raw_datasets = {
    label: load_raw_dataset(label, filename)
    for label, filename in SOURCE_FILES.items()
}

for label, df in raw_datasets.items():
    structural_check(label, df)

Loading [fund_master] from ../data/raw/01_fund_master.csv
Loading [nav_history] from ../data/raw/02_nav_history.csv
Loading [aum_by_fund_house] from ../data/raw/03_aum_by_fund_house.csv
Loading [monthly_sip_inflows] from ../data/raw/04_monthly_sip_inflows.csv
Loading [category_inflows] from ../data/raw/05_category_inflows.csv
Loading [industry_folio_count] from ../data/raw/06_industry_folio_count.csv
Loading [scheme_performance] from ../data/raw/07_scheme_performance.csv
Loading [investor_transactions] from ../data/raw/08_investor_transactions.csv
Loading [portfolio_holdings] from ../data/raw/09_portfolio_holdings.csv
Loading [benchmark_indices] from ../data/raw/10_benchmark_indices.csv

DATASET: fund_master
Shape: (40, 15)
Dtypes:
amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct 

In [3]:
fund_master_df = raw_datasets['fund_master'].copy()
nav_history_df = raw_datasets['nav_history'].copy()

fund_codes = set(
    pd.to_numeric(fund_master_df['amfi_code'], errors='coerce')
    .dropna()
    .astype(int)
    .unique()
)

nav_codes = set(
    pd.to_numeric(nav_history_df['amfi_code'], errors='coerce')
    .dropna()
    .astype(int)
    .unique()
)

missing_codes = sorted(fund_codes - nav_codes)
extra_codes = sorted(nav_codes - fund_codes)

print()
print('AMFI Coverage Verification')
print(f'Unique fund codes in 01_fund_master.csv: {len(fund_codes)}')
print(f'Unique fund codes in 02_nav_history.csv: {len(nav_codes)}')
print(f'Missing from NAV history: {missing_codes}')
print(f'Extra in NAV history: {extra_codes}')

if not missing_codes:
    print('Data Quality Check Passed: every unique fund code from 01_fund_master.csv exists in 02_nav_history.csv.')
else:
    print('Data Quality Check Failed: one or more fund codes are missing from 02_nav_history.csv.')


AMFI Coverage Verification
Unique fund codes in 01_fund_master.csv: 40
Unique fund codes in 02_nav_history.csv: 40
Missing from NAV history: []
Extra in NAV history: []
Data Quality Check Passed: every unique fund code from 01_fund_master.csv exists in 02_nav_history.csv.


## Live API Backup Capture

The AMFI live NAV API is used as a backup ingestion source for selected high-priority schemes. Each response is archived as raw JSON and as a flattened CSV backup under `../data/raw/api_backups/`.

In [4]:
SCHEME_CODES = [125497, 119551, 120503, 118632, 119092, 120841]


def fetch_and_backup_scheme(scheme_code: int, session: requests.Session) -> dict:
    url = f'https://api.mfapi.in/mf/{scheme_code}'
    response = session.get(url, timeout=30)
    response.raise_for_status()
    payload = response.json()

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    json_path = API_BACKUP_DIR / f'live_nav_{scheme_code}_{timestamp}.json'
    csv_path = API_BACKUP_DIR / f'live_nav_{scheme_code}_{timestamp}.csv'

    json_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')

    nav_rows = pd.DataFrame(payload.get('data', []))
    if not nav_rows.empty:
        nav_rows.insert(0, 'scheme_code', scheme_code)
        nav_rows.insert(1, 'scheme_name', payload.get('meta', {}).get('scheme_name', 'Unknown'))
        nav_rows.to_csv(csv_path, index=False)
    else:
        csv_path.write_text('', encoding='utf-8')

    return {
        'scheme_code': scheme_code,
        'scheme_name': payload.get('meta', {}).get('scheme_name', 'Unknown'),
        'json_path': str(json_path),
        'csv_path': str(csv_path),
        'row_count': int(len(nav_rows)),
    }


with requests.Session() as session:
    session.headers.update({'User-Agent': 'BluestockDataEngineering/1.0'})
    backup_results = [fetch_and_backup_scheme(code, session) for code in SCHEME_CODES]

backup_log = pd.DataFrame(backup_results)
print()
print('Live API Backup Results')
print(backup_log.to_string(index=False))


Live API Backup Results
 scheme_code                                                           scheme_name                                                    json_path                                                    csv_path  row_count
      125497                             SBI Small Cap Fund - Direct Plan - Growth ../data/raw/api_backups/live_nav_125497_20260603_222057.json ../data/raw/api_backups/live_nav_125497_20260603_222057.csv       3092
      119551        Aditya Birla Sun Life Banking & PSU Debt Fund  - DIRECT - IDCW ../data/raw/api_backups/live_nav_119551_20260603_222057.json ../data/raw/api_backups/live_nav_119551_20260603_222057.csv       3237
      120503                Axis ELSS Tax Saver Fund - Direct Plan - Growth Option ../data/raw/api_backups/live_nav_120503_20260603_222057.json ../data/raw/api_backups/live_nav_120503_20260603_222057.csv       3308
      118632 Nippon India Large Cap Fund - Direct Plan Growth Plan - Growth Option ../data/raw/api_backups/live_nav

## Ingestion Completion

The notebook now has a traceable raw-data inventory, a coverage check for AMFI scheme codes, and a live API backup routine ready to be rerun on demand.